In [ ]:
import numpy as np
from pyfastchem import FastChem

FASTCHEM_DIR = "/dataserver/users/formingworlds/borgmann/fastchem"

fastchem = FastChem(
    f"{FASTCHEM_DIR}/input/element_abundances/solar.dat",
    f"{FASTCHEM_DIR}/input/logK/logK.dat",
    1
)

In [ ]:
import numpy as np
from pyfastchem import FastChem, FastChemInput, FastChemOutput

# ---------------------------
# 1. Initialize FastChem
# ---------------------------
fastchem = FastChem("input/chem.dat", "input/logK.dat", 1)

# ---------------------------
# 2. Set elemental abundances (same as before)
# ---------------------------
mixing_ratios = {
    "N2": 0.78,
    "O2": 0.21,
    "CO2": 4e-4,
    "H2O": 1e-2,
    "CH4": 1e-6,
}

stoich = {
    "N2": {"N": 2},
    "O2": {"O": 2},
    "CO2": {"C": 1, "O": 2},
    "H2O": {"H": 2, "O": 1},
    "CH4": {"C": 1, "H": 4},
}

element_dict = {}
for species, frac in mixing_ratios.items():
    for elem, count in stoich[species].items():
        element_dict[elem] = element_dict.get(elem, 0) + frac * count

# normalize
total = sum(element_dict.values())
for elem in element_dict:
    element_dict[elem] /= total

# build array
n_elements = fastchem.getElementNumber()
element_abundances = np.full(n_elements, 1e-30)

for elem, val in element_dict.items():
    idx = fastchem.getElementIndex(elem)
    element_abundances[idx] = val

fastchem.setElementAbundances(element_abundances)

# ---------------------------
# 3. Create input/output objects
# ---------------------------
input_data = FastChemInput()
output_data = FastChemOutput()

# set T-P
input_data.temperature = np.array([288.0])  # K
input_data.pressure = np.array([1.0])       # bar

# ---------------------------
# 4. Run FastChem
# ---------------------------
fastchem.calcDensities(input_data, output_data)

# ---------------------------
# 5. Extract results
# ---------------------------
densities = output_data.number_densities  # shape: (n_layers, n_species)

n_species = fastchem.getGasSpeciesNumber()
total_density = np.sum(densities[0])

print("Equilibrium composition:")
for i in range(n_species):
    name = fastchem.getGasSpeciesName(i)
    mixing_ratio = densities[0][i] / total_density
    if mixing_ratio > 1e-8:
        print(f"{name:10s} {mixing_ratio:.3e}")

In [ ]:
import pyfastchem
print(dir(pyfastchem.FastChem))